# Munir (منير) — Run All

**SDAIA Academy Capstone — SDA-AIE-213: LLM Application Engineering**

This notebook is the single entry point for running and evaluating the Munir application.

### Default backend

Munir uses the deterministic mock backend by default, so no API key is required.

### Run

Use **Kernel → Restart Kernel and Run All** to execute the complete evaluation.

The notebook runs:

1. Automated tests
2. Golden-set evaluation
3. Judge calibration
4. Regression gate
5. Cost and latency replay
6. Commercial vs open-weight comparison
7. Self-host break-even analysis

The generated evaluation artifacts are written to the repository's `eval/out/` directory.

In [1]:
# ============================================================
# 1. Environment setup
# ============================================================
# Locate the repository root and make src/ importable.

from pathlib import Path
import os
import subprocess
import sys

ROOT = Path.cwd()

# If the notebook is opened from the repository root, this does nothing.
# If it is opened from the notebook/ directory, move to the repository root.
while ROOT != ROOT.parent and not (ROOT / "src" / "munir").exists():
    ROOT = ROOT.parent

if not (ROOT / "src" / "munir").exists():
    raise RuntimeError(
        "Could not find the Munir repository root. "
        "Open this notebook from inside the Munir repository."
    )

os.chdir(ROOT)

# Make both the repository root and src/ available to Python.
current_pythonpath = os.environ.get("PYTHONPATH", "")
pythonpath_parts = [str(ROOT), str(ROOT / "src")]

if current_pythonpath:
    pythonpath_parts.append(current_pythonpath)

os.environ["PYTHONPATH"] = os.pathsep.join(pythonpath_parts)

print(f"Repository: {ROOT}")
print(f"Python: {sys.executable}")
print(f"PYTHONPATH: {os.environ['PYTHONPATH']}")

Repository: /Users/loba/Munir/munir
Python: /usr/local/bin/python3.12
PYTHONPATH: /Users/loba/Munir/munir:/Users/loba/Munir/munir/src


In [3]:
# ============================================================
# 2. Verify project structure
# ============================================================

required_paths = [
    Path("src/munir"),
    Path("configs/munir.yaml"),
    Path("data"),
    Path("eval"),
    Path("scripts"),
    Path("tests"),
    Path("BENCHMARKS.md"),
    Path("EVALUATION_REPORT.md"),
    Path("DECISIONS.md"),
]

missing = [str(path) for path in required_paths if not path.exists()]

if missing:
    raise FileNotFoundError(
        "The following required project paths are missing:\n"
        + "\n".join(f"- {path}" for path in missing)
    )

print("✓ Munir project structure verified.")

✓ Munir project structure verified.


In [12]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
requirements = ROOT / "requirements.txt"

if not requirements.exists():
    raise FileNotFoundError(f"Could not find {requirements}")

print(f"Installing project dependencies from: {requirements}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements)],
    check=True,
)

print("\nRunning automated test suite...\n")

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    check=False,
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "The automated test suite failed. "
        "Review the pytest output above before continuing."
    )

print("\n✓ Automated test suite passed.")

Installing project dependencies from: /Users/loba/Munir/munir/requirements.txt
  Using cached structlog-26.1.0-py3-none-any.whl.metadata (9.7 kB)
Using cached structlog-26.1.0-py3-none-any.whl (73 kB)



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip



Running automated test suite...

......................................                                   [100%]
38 passed in 0.55s


✓ Automated test suite passed.


In [ ]:
# ============================================================
# 4. Run the evaluation pipeline
# ============================================================

commands = [
    (
        "Golden-set evaluation",
        [sys.executable, "eval/harness.py"],
    ),
    (
        "Judge calibration",
        [sys.executable, "eval/calibrate_judge.py"],
    ),
]

for name, command in commands:
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    result = subprocess.run(
        command,
        check=False,
        capture_output=True,
        text=True,
    )

    if result.stdout:
        print(result.stdout)

    if result.stderr:
        print("\n--- STDERR ---")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            f"{name} failed with exit code {result.returncode}."
        )


# ------------------------------------------------------------
# Regression gate
# The harness above generates eval/out/eval_run.json.
# We pass that freshly generated report to the gate.
# ------------------------------------------------------------

report_path = Path("eval/out/eval_run.json")

if not report_path.exists():
    raise FileNotFoundError(
        f"Expected evaluation report was not generated: {report_path}"
    )

print("\n" + "=" * 70)
print("Regression gate")
print("=" * 70)

result = subprocess.run(
    [
        sys.executable,
        "eval/gate.py",
        str(report_path),
    ],
    check=False,
    capture_output=True,
    text=True,
)

if result.stdout:
    print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"Regression gate failed with exit code {result.returncode}."
    )

print("\n✓ Evaluation pipeline completed successfully.")


Golden-set evaluation

----------------------------------------------------------------------------
eval | route=default | 120 cases | pass 120/120 (100%) | 0.08s
----------------------------------------------------------------------------
  language    ar 100% | en 100%
  intent      escalate 100% | faq 100% | service 100%
  difficulty  adversarial 100% | edge 100% | routine 100%
  risk        normal 100% | safety 100%
  p50 latency 0.7 ms

  written: /Users/loba/Munir/munir/eval/out/eval_run.json


--- STDERR ---
2026-09-15T11:30:10.904554Z [info     ] structured_extracted           attempt=1 outcome=first_try schema=guard_classification trace_id=0f16769c8588
2026-09-15T11:30:10.904918Z [info     ] llm_cost                       cache_tier= cached_tokens=0 cost_sar=0.00297 input_tokens=178 intent=unknown latency_ms=0.0 model_id=campus-flagship output_tokens=4 prompt_version= route= stage=input_guard trace_id=0f16769c8588
2026-09-15T11:30:10.905214Z [info     ] structured_extracted  

In [21]:
# ============================================================
# 5. Run cost, cache, model-comparison, and break-even evidence
# ============================================================

commands = [
    (
        "Cost and latency replay",
        [
            sys.executable,
            "scripts/replay.py",
            "--limit",
            "120",
            "--write",
        ],
    ),
    (
        "Commercial vs open-weight comparison",
        [
            sys.executable,
            "scripts/compare_models.py",
            "--limit",
            "120",
        ],
    ),
    (
        "Self-host break-even analysis",
        [
            sys.executable,
            "scripts/breakeven.py",
            "--gpu-usd-per-hour",
            "3.33",
            "--tokens-per-sec",
            "950",
            "--utilization",
            "0.50",
            "--commercial-price-per-mtok",
            "15",
            "--avg-tokens-per-request",
            "100",
        ],
    ),
]

for name, command in commands:
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    result = subprocess.run(
        command,
        check=False,
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"{name} failed with exit code {result.returncode}."
        )

print("\n✓ Cost, model comparison, and break-even analysis completed.")


Cost and latency replay


2026-09-15T11:30:36.247593Z [info     ] structured_extracted           attempt=1 outcome=first_try schema=guard_classification trace_id=d0c78bcdc7c1
2026-09-15T11:30:36.247982Z [info     ] llm_cost                       cache_tier= cached_tokens=0 cost_sar=0.00291 input_tokens=174 intent=unknown latency_ms=0.0 model_id=campus-flagship output_tokens=4 prompt_version= route= stage=input_guard trace_id=d0c78bcdc7c1
2026-09-15T11:30:36.248497Z [warning  ] structured_validation_failed   attempt=1 errors=[[]] schema=route_verdict trace_id=d0c78bcdc7c1
2026-09-15T11:30:36.248636Z [info     ] structured_extracted           attempt=2 outcome=after_repair schema=route_verdict trace_id=d0c78bcdc7c1
2026-09-15T11:30:36.248715Z [info     ] llm_cost                       cache_tier= cached_tokens=0 cost_sar=0.00219 input_tokens=126 intent=unknown latency_ms=0.0 model_id=campus-flagship output_tokens=4 prompt_version= route= stage=router trace_id=d0c78bcdc7c1
2026-09-15T11:30:36.248779Z [info     ] l


Module 6 optimisation replay
step           cost SAR   cache input     p50 ms  model calls       eval     safety
--------------------------------------------------------------------------------------------------------------
before         0.189458        70.8%        1.1           35    100.0%    100.0%
prompt         0.189458        70.8%        0.7           35    100.0%    100.0%
cache          0.117870        64.3%        0.7           31    100.0%    100.0%
cascade        0.117870        64.3%        0.7           31    100.0%    100.0%

The evaluation columns are the release verdict, not decoration.
If cost falls but the safety slice falls below 100%, do not ship the step.

written: /Users/loba/Munir/munir/eval/out/cost_optimization_comparison.json

Commercial vs open-weight comparison


2026-09-15T11:30:38.814801Z [info     ] structured_extracted           attempt=1 outcome=first_try schema=guard_classification trace_id=c45cea048d4c
2026-09-15T11:30:38.815672Z [info     ] llm_cost                       cache_tier= cached_tokens=0 cost_sar=0.00297 input_tokens=178 intent=unknown latency_ms=0.0 model_id=campus-flagship output_tokens=4 prompt_version= route= stage=input_guard trace_id=c45cea048d4c
2026-09-15T11:30:38.816351Z [info     ] structured_extracted           attempt=1 outcome=first_try schema=route_verdict trace_id=c45cea048d4c
2026-09-15T11:30:38.816475Z [info     ] llm_cost                       cache_tier= cached_tokens=0 cost_sar=0.0021 input_tokens=130 intent=unknown latency_ms=0.0 model_id=campus-flagship output_tokens=2 prompt_version= route= stage=router trace_id=c45cea048d4c
2026-09-15T11:30:38.816605Z [info     ] routed                         intent=faq prompt_version=route_intent.v1 trace_id=c45cea048d4c
2026-09-15T11:30:38.821027Z [info     ] llm_co


COMMERCIAL VS OPEN-WEIGHT
primary      pass=119/120 (99%) | cost=1.6338 SAR | calls=335 | benchmark tok/s=950.00
  observed tok/s: 0.00
  language   : ar=100% | en=99%
  intent     : escalate=100% | faq=100% | service=97%
  difficulty : adversarial=100% | edge=100% | routine=98%
  risk       : normal=99% | safety=100%
open_weight  pass=117/120 (98%) | cost=0.1686 SAR | calls=335 | benchmark tok/s=950.00
  observed tok/s: 0.00
  language   : ar=98% | en=97%
  intent     : escalate=100% | faq=97% | service=97%
  difficulty : adversarial=100% | edge=93% | routine=98%
  risk       : normal=97% | safety=100%

SLICE DELTAS (second route minus first route)
  language    ar           -2%
  language    en           -1%
  intent      escalate     +0%
  intent      faq          -3%
  intent      service      +0%
  difficulty  adversarial  +0%
  difficulty  edge         -7%
  difficulty  routine      +0%
  risk        normal       -2%
  risk        safety       +0%

written: /Users/loba/Munir/mun

In [22]:
# ============================================================
# 6. Verify generated evaluation artifacts
# ============================================================

print("=" * 70)
print("GENERATED EVIDENCE")
print("=" * 70)

files_to_check = [
    Path("BENCHMARKS.md"),
    Path("EVALUATION_REPORT.md"),
    Path("DECISIONS.md"),
    Path("eval/out/cost_optimization_comparison.json"),
    Path("eval/out/commercial_vs_open_weight.json"),
]

for path in files_to_check:
    if path.exists():
        print(f"✓ {path}")
    else:
        print(f"⚠ Missing: {path}")

print("\nRun All completed.")

GENERATED EVIDENCE
✓ BENCHMARKS.md
✓ EVALUATION_REPORT.md
✓ DECISIONS.md
✓ eval/out/cost_optimization_comparison.json
✓ eval/out/commercial_vs_open_weight.json

Run All completed.
